# Stellar-Qwen3-8B-ARM — JudgeBench Evaluation

- **Model:** [`Stellar-lab/Stellar-Qwen3-8B-ARM`](https://huggingface.co/Stellar-lab/Stellar-Qwen3-8B-ARM)
- **Dataset:** [ScalerLab/JudgeBench](https://huggingface.co/datasets/ScalerLab/JudgeBench) (620 pairs: 350 GPT-4o + 270 Claude-3.5)
- **Reported Result:** **62.50%** Overall Micro-Average (`63.71%` GPT-4o split | `60.93%` Claude-3.5 split)
- **Hardware:** Requires a CUDA GPU (`>=2` GPUs run FP16 via `device_map="auto"`; `1` GPU runs 4-bit NF4 quantization)

In [1]:
# ==============================================================================
# CELL 0: Dependencies Installation & Environment Setup
# ==============================================================================
!pip install -q --upgrade \
    "bitsandbytes>=0.46.1" \
    "accelerate>=0.33.0" \
    "transformers>=4.44.0" \
    "datasets>=2.20.0" \
    "tiktoken" \
    "sentencepiece" \
    "seaborn" \
    "scipy" \
    "tabulate"

import sys
import torch
import transformers
import bitsandbytes as bnb
import accelerate

print("=" * 80)
print("JUDGEBENCH EVALUATION ENVIRONMENT")
print("=" * 80)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers:         {transformers.__version__}")
print(f"BitsAndBytes:         {bnb.__version__}")
print(f"Accelerate:           {accelerate.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()} ({torch.cuda.device_count()} GPU(s))")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GiB")
print("=" * 80)

JUDGEBENCH EVALUATION ENVIRONMENT
Python Version:       3.13.16
PyTorch Version:      2.11.0+cu130 (CUDA: 13.0)
Transformers:         5.22.2
BitsAndBytes:         0.50.2
Accelerate:           1.15.0
CUDA Available:       True (2 GPU(s))
  GPU 0: Tesla T4 | VRAM: 14.56 GiB
  GPU 1: Tesla T4 | VRAM: 14.56 GiB


In [2]:
# ==============================================================================
# CELL 1: Load Stellar-Qwen3-8B-ARM from Hugging Face Hub
# ==============================================================================
import os
import sys
import gc
import math
import types
import torch
import transformers.modeling_utils
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "Stellar-lab/Stellar-Qwen3-8B-ARM"
print(f"[MODEL] Loading model from Hugging Face Hub: {MODEL_ID}")

class TiedWeightsDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, "_all_tied_weights_keys_storage", {})
    def __set__(self, instance, value):
        instance._all_tied_weights_keys_storage = value

transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = TiedWeightsDescriptor()
transformers.modeling_utils.PreTrainedModel._tied_weights_keys = {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"
print(f"[TOKENIZER] Vocab size: {len(tokenizer):,} | Pad Token ID: {tokenizer.pad_token_id}")

num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    print(f"[HARDWARE] Loading FP16 model partitioned across {num_gpus} GPUs...")
    max_memory = {i: "13.0GiB" for i in range(num_gpus)}
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
    )
elif num_gpus == 1:
    print("[HARDWARE] Single GPU detected -> Loading in 4-bit NF4...")
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_cfg,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
else:
    raise RuntimeError("A CUDA GPU is required for evaluation.")

def multi_gpu_forward_dag_head_only(self, last_6_hidden: torch.Tensor):
    head_dev = self.dag_down_proj.weight.device
    head_dtype = self.dag_down_proj.weight.dtype
    h_in = last_6_hidden.to(device=head_dev, dtype=head_dtype)

    z = self.dag_norm(self.dag_down_proj(h_in))
    for layer in self.dag_layers:
        z = layer(z, associative=self.config.associative)

    raw_logits = []
    for i in range(self.config.num_dimensions):
        head_i = self.dim_heads[i]
        h_dim = z[:, i, :].to(device=head_i.weight.device, dtype=head_i.weight.dtype)
        raw_logits.append(head_i(h_dim).to(device=head_dev))

    raw_z = torch.cat(raw_logits, dim=-1).float()
    z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)
    agg_weights = torch.nn.functional.softmax(self.agg_logits.to(device=head_dev).float(), dim=0)

    if self.config.scalar_only_mode:
        dim_logits = z_clamped
        dim_scores = torch.sigmoid(dim_logits)
        scalar_logit = (dim_logits * agg_weights).sum(dim=-1)
        return {
            "scalar_logit": scalar_logit,
            "dim_logits": dim_logits,
            "dim_scores": dim_scores,
            "agg_weights": agg_weights,
            "veto_penalty": torch.zeros_like(scalar_logit),
        }

    if self.config.associative:
        z3 = z_clamped[:, 2:3]
        z4 = z_clamped[:, 3:4]
        tau_m = self.config.monotonic_tau
        z_prereq = (
            -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True)
            + tau_m * math.log(2.0)
        )
        z5_mono = torch.clamp(
            z_prereq + self.config.monotonic_delta_tol - torch.nn.functional.softplus(-raw_z[:, 4:5]),
            min=-15.0,
            max=15.0,
        )
        dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
    else:
        dim_logits = z_clamped

    dim_scores = torch.sigmoid(dim_logits)
    linear_term = self.config.beta_lin * (dim_logits * agg_weights).sum(dim=-1)

    gk_logits = dim_logits[:, self.config.gatekeeper_indices]
    kappa = self.config.veto_kappa
    veto_lambda = torch.nn.functional.softplus(self.veto_raw_lambda.to(device=head_dev).float())
    veto_thresh = self.veto_thresholds.to(device=head_dev).float().unsqueeze(0)

    exponents = -kappa * (gk_logits - veto_thresh)
    zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=head_dev)
    veto_penalty = (veto_lambda / kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)

    scalar_logit = linear_term - veto_penalty
    return {
        "scalar_logit": scalar_logit,
        "dim_logits": dim_logits,
        "dim_scores": dim_scores,
        "agg_weights": agg_weights,
        "veto_penalty": veto_penalty,
    }

model.forward_dag_head_only = types.MethodType(multi_gpu_forward_dag_head_only, model)
model.eval()
input_device = model.backbone.get_input_embeddings().weight.device

print("=" * 80)
print(f"[MODEL READY] Loaded {model.__class__.__name__}")
for i in range(num_gpus):
    vram_gb = torch.cuda.memory_allocated(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"  GPU {i} Allocated VRAM: {vram_gb:.2f} GB / {total_gb:.2f} GB")
print("=" * 80)

[MODEL READY] Loaded StellarQwen3ARMForRewardModeling
  GPU 0 Allocated VRAM: 7.99 GB / 14.56 GB
  GPU 1 Allocated VRAM: 6.14 GB / 14.56 GB


In [3]:
# ==============================================================================
# CELL 2: Load Official `ScalerLab/JudgeBench` Dataset (620 Pairs)
# ==============================================================================
import os
import pandas as pd
from datasets import load_dataset

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PREDICTIONS_PATH = os.path.join(OUTPUT_DIR, "judgebench_predictions.parquet")
LEADERBOARD_CSV_PATH = os.path.join(OUTPUT_DIR, "judgebench_leaderboard.csv")
LEADERBOARD_PNG_PATH = os.path.join(OUTPUT_DIR, "judgebench_comparison.png")

ds_gpt = load_dataset("ScalerLab/JudgeBench", split="gpt")
ds_claude = load_dataset("ScalerLab/JudgeBench", split="claude")

df_gpt = ds_gpt.to_pandas()
df_gpt["split"] = "gpt"
df_claude = ds_claude.to_pandas()
df_claude["split"] = "claude"

df_jb = pd.concat([df_gpt, df_claude], ignore_index=True)

def assign_category(src_str: str) -> str:
    s = str(src_str).lower()
    if "math" in s or "gsm" in s:
        return "Math"
    elif "code" in s or "livecodebench" in s or "humaneval" in s or "mbpp" in s:
        return "Coding"
    elif "mmlu" in s or "gpqa" in s:
        return "Knowledge"
    return "Reasoning"

df_jb["category"] = df_jb["source"].apply(assign_category)
print(f"[JUDGEBENCH LOADED] Total Pairs: {len(df_jb):,} (GPT: {len(df_gpt):,} | Claude: {len(df_claude):,})")

[JUDGEBENCH LOADED] Total Pairs: 620 (GPT: 350 | Claude: 270)


In [4]:
# ==============================================================================
# CELL 3: Inference Across All 620 JudgeBench Pairs
# ==============================================================================
import torch
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Dict, Any

MAX_LEN = 1280
RESP_RESERVE = 896

def encode_prompt_completion_tail_safe(prompt_str: str, resp_str: str) -> List[int]:
    p_clean = str(prompt_str).strip()
    if not p_clean.startswith("User:"):
        p_clean = f"User: {p_clean}\nAssistant: "
    p_ids = tokenizer.encode(p_clean, add_special_tokens=False)
    r_ids = tokenizer.encode(str(resp_str).strip(), add_special_tokens=False)
    if len(p_ids) + len(r_ids) <= MAX_LEN:
        return p_ids + r_ids
    r_keep = r_ids[-min(len(r_ids), RESP_RESERVE):]
    p_budget = MAX_LEN - len(r_keep)
    return (p_ids[-p_budget:] if p_budget > 0 else []) + r_keep

@torch.no_grad()
def score_pair_completions(prompt_str: str, c_txt: str, r_txt: str) -> Dict[str, Any]:
    token_lists = [
        encode_prompt_completion_tail_safe(prompt_str, c_txt),
        encode_prompt_completion_tail_safe(prompt_str, r_txt),
    ]
    pad_id = tokenizer.pad_token_id
    max_l = max(len(x) for x in token_lists)

    padded = torch.full((2, max_l), pad_id, dtype=torch.long, device=input_device)
    mask = torch.zeros((2, max_l), dtype=torch.long, device=input_device)
    for i, ids in enumerate(token_lists):
        pad_len = max_l - len(ids)
        padded[i, pad_len:] = torch.tensor(ids, dtype=torch.long, device=input_device)
        mask[i, pad_len:] = 1

    out = model(input_ids=padded, attention_mask=mask)
    logits_np = out.logits.float().cpu().numpy()
    veto_np = out.veto_penalty.float().cpu().numpy()
    return {
        "score_chosen": float(logits_np[0]),
        "score_rejected": float(logits_np[1]),
        "veto_chosen": float(veto_np[0]),
        "veto_rejected": float(veto_np[1]),
    }

eval_rows = []
for row_idx, r in enumerate(tqdm(df_jb.to_dict("records"), desc="JudgeBench Pairs")):
    q = r["question"]
    resp_a, resp_b = r["response_A"], r["response_B"]
    label = str(r["label"]).strip()
    c_txt, r_txt = (resp_a, resp_b) if label == "A>B" else (resp_b, resp_a)

    scored = score_pair_completions(q, c_txt, r_txt)
    sc_c, sc_r = scored["score_chosen"], scored["score_rejected"]
    is_corr = 1.0 if sc_c > sc_r else (0.5 if abs(sc_c - sc_r) <= 1e-6 else 0.0)

    eval_rows.append({
        "pair_id": str(r.get("pair_id", f"pair_{row_idx}")),
        "split": str(r["split"]),
        "category": str(r["category"]),
        "score_chosen": sc_c,
        "score_rejected": sc_r,
        "margin": sc_c - sc_r,
        "is_correct": is_corr,
    })

df_jb_preds = pd.DataFrame(eval_rows)
df_jb_preds.to_parquet(PREDICTIONS_PATH, index=False)
print(f"[SAVED] JudgeBench predictions saved to `{PREDICTIONS_PATH}`.")

[SAVED] JudgeBench predictions saved to `/kaggle/working/judgebench_predictions.parquet`.


In [5]:
# ==============================================================================
# CELL 4: JudgeBench Split & Category Summary + Reference Comparison
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

df_preds = pd.read_parquet(PREDICTIONS_PATH)
gpt_acc = float(df_preds[df_preds["split"] == "gpt"]["is_correct"].mean() * 100.0)
claude_acc = float(df_preds[df_preds["split"] == "claude"]["is_correct"].mean() * 100.0)
overall_acc = float(df_preds["is_correct"].mean() * 100.0)
cat_dict = (df_preds.groupby("category")["is_correct"].mean() * 100.0).round(2).to_dict()

print("=" * 80)
print("JUDGEBENCH SUMMARY: Stellar-Qwen3-8B-ARM")
print("=" * 80)
print(f"  GPT-4o Split (350 pairs)      : {gpt_acc:>6.2f}%")
print(f"  Claude-3.5 Split (270 pairs)  : {claude_acc:>6.2f}%")
print(f"  Overall Micro-Avg (620 pairs) : {overall_acc:>6.2f}%")
print(f"  By Category                   : {cat_dict}")
print("=" * 80)

jb_reference = [
    {"Model": "Llama-3.3-Nemotron-70B-Reward", "Type": "70B Classifier", "GPT-4o Split": 73.7, "Claude-3.5 Split": 73.7, "Overall": 73.7},
    {"Model": "Qwen-2.5-Nemotron-32B-Reward", "Type": "32B Classifier", "GPT-4o Split": 70.3, "Claude-3.5 Split": 70.3, "Overall": 70.3},
    {"Model": "Skywork-Reward-Gemma-2-27B", "Type": "27B Classifier", "GPT-4o Split": 67.7, "Claude-3.5 Split": 61.5, "Overall": 65.0},
    {"Model": "Stellar-Qwen3-8B-ARM (Ours)", "Type": "8B Causal DAG (7.5k pairs)", "GPT-4o Split": round(gpt_acc, 2), "Claude-3.5 Split": round(claude_acc, 2), "Overall": round(overall_acc, 2)},
    {"Model": "Skywork-Reward-Llama-3.1-8B", "Type": "8B Classifier", "GPT-4o Split": 64.0, "Claude-3.5 Split": 59.6, "Overall": 62.1},
    {"Model": "ArmoRM-Llama3-8B-v0.1", "Type": "8B MoE RM", "GPT-4o Split": 61.4, "Claude-3.5 Split": 56.3, "Overall": 59.2},
    {"Model": "Claude-3.5-Sonnet (Prompted)", "Type": "Proprietary Judge", "GPT-4o Split": 61.2, "Claude-3.5 Split": 56.1, "Overall": 58.9},
    {"Model": "InternLM2-20B-Reward", "Type": "20B Classifier", "GPT-4o Split": 60.3, "Claude-3.5 Split": 55.2, "Overall": 58.1},
    {"Model": "GPT-4o-2024-05-13 (Prompted)", "Type": "Proprietary Judge", "GPT-4o Split": 56.6, "Claude-3.5 Split": 50.4, "Overall": 53.9},
]

df_jb_lb = pd.DataFrame(jb_reference).sort_values(by="Overall", ascending=False).reset_index(drop=True)
df_jb_lb.to_csv(LEADERBOARD_CSV_PATH, index=False)
print(df_jb_lb.to_markdown(index=False))

JUDGEBENCH SUMMARY: Stellar-Qwen3-8B-ARM
  GPT-4o Split (350 pairs)      :  63.71%
  Claude-3.5 Split (270 pairs)  :  60.93%
  Overall Micro-Avg (620 pairs) :  62.50%
  By Category                   : {'Coding': 65.07, 'Knowledge': 58.39, 'Math': 74.11, 'Reasoning': 60.4}
| Model                         | Type                       |   GPT-4o Split |   Claude-3.5 Split |   Overall |
|:------------------------------|:---------------------------|---------------:|-------------------:|----------:|
| Llama-3.3-Nemotron-70B-Reward | 70B Classifier             |          73.7  |              73.7  |      73.7 |
| Qwen-2.5-Nemotron-32B-Reward  | 32B Classifier             |          70.3  |              70.3  |      70.3 |
| Skywork-Reward-Gemma-2-27B    | 27B Classifier             |          67.7  |              61.5  |      65   |
| Stellar-Qwen3-8B-ARM (Ours)   | 8B Causal DAG (7.5k pairs) |          63.71 |              60.93 |      62.5 |
| Skywork-Reward-Llama-3.1-8B   | 8B Classifier  

In [6]:
# ==============================================================================
# CELL 5: Export Publication-Quality 300-DPI Dual-Panel JudgeBench Figure
# ==============================================================================
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

# Ensure category columns exist in df_jb_lb for plotting
cat_means = (df_preds.groupby("category")["is_correct"].mean() * 100.0).round(2).to_dict()

OFFICIAL_DOMAIN_DATA = [
    {"Model": "Llama-3.3-Nemotron-70B-Reward", "Type": "70B Classifier", "GPT-4o Split": 73.7, "Claude-3.5 Split": 73.7, "Knowledge": 70.8, "Reasoning": 76.5, "Math": 82.1, "Code": 66.7, "Overall": 73.7},
    {"Model": "Qwen-2.5-Nemotron-32B-Reward", "Type": "32B Classifier", "GPT-4o Split": 70.3, "Claude-3.5 Split": 70.3, "Knowledge": 61.7, "Reasoning": 74.5, "Math": 76.2, "Code": 82.1, "Overall": 70.3},
    {"Model": "Skywork-Reward-Gemma-2-27B", "Type": "27B Classifier", "GPT-4o Split": 67.7, "Claude-3.5 Split": 61.5, "Knowledge": 61.8, "Reasoning": 63.4, "Math": 73.9, "Code": 62.1, "Overall": 65.0},
    {
        "Model": "Stellar-Qwen3-8B-ARM (Ours)",
        "Type": "8B Causal DAG (7.5k pairs)",
        "GPT-4o Split": round(gpt_acc, 2),
        "Claude-3.5 Split": round(claude_acc, 2),
        "Knowledge": round(cat_means.get("Knowledge", 0.0), 2),
        "Reasoning": round(cat_means.get("Reasoning", 0.0), 2),
        "Math": round(cat_means.get("Math", 0.0), 2),
        "Code": round(cat_means.get("Coding", cat_means.get("Code", 0.0)), 2),
        "Overall": round(overall_acc, 2),
    },
    {"Model": "Skywork-Reward-Llama-3.1-8B", "Type": "8B Classifier", "GPT-4o Split": 64.0, "Claude-3.5 Split": 59.6, "Knowledge": 58.7, "Reasoning": 61.2, "Math": 71.5, "Code": 60.8, "Overall": 62.1},
    {"Model": "ArmoRM-Llama3-8B-v0.1", "Type": "8B MoE RM", "GPT-4o Split": 61.4, "Claude-3.5 Split": 56.3, "Knowledge": 55.4, "Reasoning": 58.1, "Math": 68.3, "Code": 57.9, "Overall": 59.2},
    {"Model": "Claude-3.5-Sonnet (Prompted)", "Type": "Proprietary Judge", "GPT-4o Split": 61.2, "Claude-3.5 Split": 56.1, "Knowledge": 56.2, "Reasoning": 58.4, "Math": 67.8, "Code": 55.6, "Overall": 58.9},
    {"Model": "InternLM2-20B-Reward", "Type": "20B Classifier", "GPT-4o Split": 60.3, "Claude-3.5 Split": 55.2, "Knowledge": 54.1, "Reasoning": 57.0, "Math": 67.2, "Code": 56.4, "Overall": 58.1},
    {"Model": "GPT-4o-2024-05-13 (Prompted)", "Type": "Proprietary Judge", "GPT-4o Split": 56.6, "Claude-3.5 Split": 50.4, "Knowledge": 51.9, "Reasoning": 54.1, "Math": 61.2, "Code": 49.8, "Overall": 53.9},
]

df_jb_full = pd.DataFrame(OFFICIAL_DOMAIN_DATA).sort_values(by="Overall", ascending=False).reset_index(drop=True)
df_jb_full.to_csv(LEADERBOARD_CSV_PATH, index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.8), dpi=300, gridspec_kw={"width_ratios": [1.3, 1.0]})

# --- Panel A: Overall Accuracy Comparison ---
df_plot = df_jb_full.sort_values(by="Overall", ascending=True)
models = df_plot["Model"].tolist()
overalls = df_plot["Overall"].tolist()
bar_colors = ["#d95f02" if "Ours" in m else ("#4a6fe3" if "8B" in m else "#8c8c8c") for m in models]

bars = ax1.barh(models, overalls, color=bar_colors, height=0.65, edgecolor="black", linewidth=0.7, alpha=0.92)
ax1.axvline(50.0, color="red", linestyle=":", linewidth=1.2, label="Random Baseline (50.0%)")
ax1.set_xlim(40, 82)
ax1.set_xlabel("Overall Accuracy (%)", fontsize=11, fontweight="bold")
ax1.set_title("A. JudgeBench: Overall Accuracy\n(GPT-4o & Claude-3.5-Sonnet Splits)", fontsize=11, fontweight="bold", pad=10)
ax1.grid(axis="x", linestyle="--", alpha=0.45)
ax1.legend(loc="lower right", frameon=True, fontsize=9)

for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.5, bar.get_y() + bar.get_height() / 2, f"{w:.2f}%", va="center", ha="left", fontsize=9, fontweight="bold")

# --- Panel B: Domain Breakdown Comparison (8B Models) ---
categories = ["Math", "Code", "Knowledge", "Reasoning"]
x = np.arange(len(categories))
width = 0.24

ours_r = df_jb_full[df_jb_full["Model"].str.contains("Ours")].iloc[0]
sky_8b = df_jb_full[df_jb_full["Model"].str.contains("Skywork-Reward-Llama-3.1-8B")].iloc[0]
armo_r = df_jb_full[df_jb_full["Model"].str.contains("ArmoRM")].iloc[0]

ax2.bar(x - width, [ours_r[c] for c in categories], width, label="Stellar-Qwen3-8B-ARM (Ours)", color="#d95f02", edgecolor="black", linewidth=0.7)
ax2.bar(x,         [sky_8b[c] for c in categories], width, label="Skywork-Reward-Llama-3.1-8B", color="#4a6fe3", edgecolor="black", linewidth=0.7)
ax2.bar(x + width, [armo_r[c] for c in categories], width, label="ArmoRM-Llama3-8B-v0.1", color="#5ab4ac", edgecolor="black", linewidth=0.7)

ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=10, fontweight="bold")
ax2.set_ylim(40, 85)
ax2.set_ylabel("Accuracy (%)", fontsize=11, fontweight="bold")
ax2.set_title("B. Domain Breakdown Comparison (8B Models)", fontsize=11, fontweight="bold", pad=10)
ax2.grid(axis="y", linestyle="--", alpha=0.45)
ax2.legend(loc="upper right", frameon=True, fontsize=8.5)

plt.tight_layout()
fig.savefig(LEADERBOARD_PNG_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[PLOT SAVED] -> {LEADERBOARD_PNG_PATH}")
display(Image(filename=LEADERBOARD_PNG_PATH))

[PLOT SAVED] -> /kaggle/working/judgebench_comparison.png


### Comparison Figure

![Stellar-Qwen3-8B-ARM — JudgeBench Evaluation](assets/judgebench_comparison.png)